# Example 7.8.3 / 1: Copy a Pool into a Table and Check It

Companion notebook for **7.8.3 Exporting Several Solutions** in the AMPL lab Documentation, section "Copy a Pool into a Table and Check It". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Copy a Pool into a Table and Check It

Each `solution poolK.sol` command replaces the values in the AMPL variables, so the loop copies
each solution into plain Python records before loading the next. One row per solution and item
keeps the AMPL index in its own column instead of packing it into a string:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;  # weight capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;  # volume capacity

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
import pandas as pd

ampl.option["solver_msg"] = 0
ampl.solve(solver="gcg", gcg_options="sol:stub=pool", verbose=False)
records = []
for k in range(1, int(ampl.get_value("Initial.nsol")) + 1):
    ampl.eval(f"solution pool{k}.sol;")
    objective = ampl.obj["Total_Value"].value()
    for item, take in ampl.var["Take"].to_dict().items():  # copy before the next solution
        records.append({"solution_id": k, "objective": objective, "item": item,
                        "take": round(take)})
pool = pd.DataFrame(records)
pool.to_csv("pool.csv", index=False)
print(pool[pool["take"] == 1].groupby("solution_id")
      .agg(objective=("objective", "first"), items=("item", " ".join)))

             objective        items
solution_id                        
1                153.0  i1 i2 i5 i6
2                145.0     i2 i6 i8
3                142.0     i3 i6 i8
4                138.0     i6 i7 i8


### Pure AMPL: data.dat and model.run

`pool.csv` has one row for every solution and item, 40 rows here; the printed summary keeps only
the items taken. The empty knapsack, solution 5, has no row with `take = 1`, so it is missing from
the summary but present in the file: record every instantiated member, or state that missing rows
mean zero.

The same in pure AMPL, with the same `model.mod`. `printf` writes the same CSV rows, and `> pool.csv`
after a `printf` would send them to the file instead of the screen:

In [6]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver gcg;
option solver_msg 0;
option gcg_options 'sol:stub=pool';
solve;
printf "solution_id,objective,item,take\n";
for {k in 1..Initial.nsol} {
    solution ("pool" & k & ".sol");
    printf {i in ITEMS: Take[i] > 0.5} "%d,%g,%s,%d\n", k, Total_Value, i, round(Take[i]);
}

GCG 4.0.2:   sol:stub = pool

suffix nsol OUT;
suffix npool OUT;
solution_id,objective,item,take
1,153,i1,1
1,153,i2,1
1,153,i5,1
1,153,i6,1
2,145,i2,1
2,145,i6,1
2,145,i8,1
3,142,i3,1
3,142,i6,1
3,142,i8,1
4,138,i6,1
4,138,i7,1
4,138,i8,1


### Another way in amplpy

This version prints only the items taken, to keep the listing short.

Then check the file: read it back and recompute each objective from its rows and the item values.
A mismatch reveals a lost index column, a row copied from the wrong solution, or values rounded
beyond the feasibility tolerance. With the same `model.mod` and `data.py`, after the export above:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [11]:
# run.py
import pandas as pd

saved = pd.read_csv("pool.csv")
value = ampl.param["value"].to_dict()
saved["contribution"] = saved["item"].map(value) * saved["take"]
check = saved.groupby("solution_id").agg(objective=("objective", "first"),
                                         recomputed=("contribution", "sum"))
print(check)
print("all match:", bool((check["objective"] == check["recomputed"]).all()))

             objective  recomputed
solution_id                       
1                153.0         153
2                145.0         145
3                142.0         142
4                138.0         138
5                  0.0           0
all match: True
